# 第1课：FASTA、张量与一次真实梯度更新

这是CPU教学notebook。基础环境只需NumPy；PyTorch对照可选。不联网、不下载模型、不读写外部文件。

**范围：** 这里使用人为构造的短序列和教学token字典；不代表PF00194数据，也不是DPLM训练。先预测输出，再执行单元；练习题由你亲手修改和作答。


In [ ]:
import numpy as np

print('NumPy:', np.__version__)
AA = 'ACDEFGHIKLMNPQRSTVWY'
PAD, MASK = 0, 1  # 教学编号，不是正式DPLM tokenizer编号
vocab = {aa: i + 2 for i, aa in enumerate(AA)}
id_to_token = {PAD: '<pad>', MASK: '<mask>', **{i: aa for aa, i in vocab.items()}}
print('教学词表大小:', len(vocab) + 2)


## 1. 读取一个小FASTA

FASTA的`>`行保存ID/描述，随后一行或多行拼成序列。示例解析器故意保持短小，只接受20种标准氨基酸，并拒绝空记录或重复ID。正式生产数据还需要更多来源和质量审计。


In [ ]:
toy_fasta = '''>toy_1 synthetic teaching sequence
ACDE
FG
>toy_2 synthetic teaching sequence
MKWV
'''

def parse_fasta(text):
    records, current_id, parts = [], None, []
    def finish():
        if current_id is None:
            return
        sequence = ''.join(parts).upper()
        if not sequence:
            raise ValueError(f'空序列: {current_id}')
        illegal = set(sequence) - set(AA)
        if illegal:
            raise ValueError(f'非法字母 {sorted(illegal)}: {current_id}')
        records.append((current_id, sequence))
    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            continue
        if line.startswith('>'):
            finish()
            header = line[1:].split()
            if not header:
                raise ValueError('缺少FASTA ID')
            current_id, parts = header[0], []
        else:
            if current_id is None:
                raise ValueError('序列出现在header之前')
            parts.append(line)
    finish()
    if not records:
        raise ValueError('没有FASTA记录')
    ids = [rid for rid, _ in records]
    if len(ids) != len(set(ids)):
        raise ValueError('重复ID')
    return records

records = parse_fasta(toy_fasta)
print(records)
lengths = np.array([len(sequence) for _, sequence in records])
print('长度:', lengths, '平均:', lengths.mean())


### 练习A（请自己做）

1. 新增第三条不同长度的记录，先在纸上预测统计值，再运行。
2. 添加非法字符`?`、空记录和重复ID，分别说明应在哪里被拒绝。可以在新单元用`try/except`观察。
3. 不复制上面的统计表达式，独立写一个返回条数、最短、最长、平均长度的函数。
4. 两个不同ID的序列完全相同时，这个解析器是否会拒绝？正式数据流程还需要什么检查？


## 2. 变成batch，补齐长度

`targets`形状为`B × L`，B是序列条数、L是当前batch最长长度。PAD只用于补齐，不是一个真实氨基酸。MASK表示某个原本存在的氨基酸被遮住，两者用途不同。


In [ ]:
batch_size, max_length = len(records), int(lengths.max())
targets = np.full((batch_size, max_length), PAD, dtype=np.int64)
for row, (_, sequence) in enumerate(records):
    targets[row, :len(sequence)] = [vocab[aa] for aa in sequence]
valid = targets != PAD
print('targets shape:', targets.shape)
print(targets)
print('真实氨基酸位置:', valid)
assert valid.sum() == lengths.sum()


## 3. 固定随机种子，破坏部分真实位置

这里固定0.5的掩码概率，只是为了观察数据。DPLM会采样时间步，让破坏程度变化，并使用正式特殊token规则；这个单元不复现完整扩散目标。


In [ ]:
rng = np.random.default_rng(42)
loss_mask = (rng.random(targets.shape) < 0.5) & valid
if not loss_mask.any():
    loss_mask[tuple(np.argwhere(valid)[0])] = True
inputs = targets.copy()
inputs[loss_mask] = MASK
print('输入:', inputs)
print('目标:', targets)
print('loss mask:', loss_mask)
assert not np.any(loss_mask & ~valid)
assert np.all(inputs[~valid] == PAD)


### 练习B（请自己做）

- 添加第三条序列后，逐一预测`targets`、`valid`、`loss_mask`的shape。
- 将种子改为43，再回到42，记录哪些变化可复现。
- 不执行代码，解释为什么`inputs`与`targets`不能是同一个被原地修改的数组。
- 如果一个batch恰好没有mask，交叉熵平均值的分母会发生什么？


## 4. NumPy计算交叉熵与手动梯度

假设每个位置直接拥有一组可训练logits，形状为`B × L × V`。V是教学词表大小。我们只优化这些独立logits，**没有Transformer，也没有共享参数，不能从这里得到蛋白生成能力**。

掩码位置的交叉熵为`-log(正确token的预测概率)`。下面用稳定softmax计算其平均值和梯度；未掩码位置梯度为零。


In [ ]:
vocab_size = len(vocab) + 2
logits = np.random.default_rng(7).normal(0, 0.1, (*targets.shape, vocab_size))

def masked_ce_and_gradient(values, labels, selected):
    count = int(selected.sum())
    if count == 0:
        raise ValueError('教学例子要求至少一个mask位置')
    shifted = values - values.max(axis=-1, keepdims=True)
    exp_values = np.exp(shifted)
    probabilities = exp_values / exp_values.sum(axis=-1, keepdims=True)
    correct = np.take_along_axis(probabilities, labels[..., None], axis=-1)[..., 0]
    loss = float((-np.log(correct[selected])).mean())
    gradient = probabilities.copy()
    for row, col in np.ndindex(labels.shape):
        gradient[row, col, labels[row, col]] -= 1.0
    gradient *= selected[..., None] / count
    return loss, gradient

initial_loss, initial_gradient = masked_ce_and_gradient(logits, targets, loss_mask)
assert np.all(initial_gradient[~loss_mask] == 0)
print('logits shape:', logits.shape)
print('初始loss:', initial_loss)
print('梯度范数:', np.linalg.norm(initial_gradient))


## 5. 用有限差分核对一个梯度，再执行更新

有限差分只做教学核查，不应对150M个参数这样计算。更新公式中的学习率控制移动幅度；计算梯度和实际更新是两个动作。


In [ ]:
row, col = map(int, np.argwhere(loss_mask)[0])
check_index = (row, col, int(targets[row, col]))
epsilon = 1e-5
plus, minus = logits.copy(), logits.copy()
plus[check_index] += epsilon
minus[check_index] -= epsilon
finite_difference = (
    masked_ce_and_gradient(plus, targets, loss_mask)[0]
    - masked_ce_and_gradient(minus, targets, loss_mask)[0]
) / (2 * epsilon)
assert np.isclose(finite_difference, initial_gradient[check_index], atol=1e-6)
print('梯度检查通过')

trained_logits = logits.copy()
learning_rate = 0.5
loss_history = []
for step in range(30):
    loss, gradient = masked_ce_and_gradient(trained_logits, targets, loss_mask)
    loss_history.append(loss)
    trained_logits -= learning_rate * gradient
final_loss, _ = masked_ce_and_gradient(trained_logits, targets, loss_mask)
assert final_loss < initial_loss
print('loss:', initial_loss, '→', final_loss)
print('未选择位置保持不变:', np.array_equal(trained_logits[~loss_mask], logits[~loss_mask]))


## 6. 可选：用PyTorch核对自动求导

未安装PyTorch时正常跳过。NumPy部分已经演示实际参数更新。即使这里有自动求导，它仍然只是独立logits玩具，不能写成DPLM微调结果。


In [ ]:
try:
    import torch
except ImportError:
    print('未安装PyTorch：跳过可选对照，CPU NumPy练习已完成。')
else:
    tensor_logits = torch.tensor(logits, dtype=torch.float64, requires_grad=True)
    tensor_targets = torch.tensor(targets, dtype=torch.long)
    tensor_mask = torch.tensor(loss_mask, dtype=torch.bool)
    token_losses = torch.nn.functional.cross_entropy(
        tensor_logits.reshape(-1, vocab_size), tensor_targets.reshape(-1), reduction='none'
    ).reshape(targets.shape)
    torch_loss = token_losses[tensor_mask].mean()
    torch_loss.backward()
    assert np.isclose(torch_loss.item(), initial_loss)
    assert np.allclose(tensor_logits.grad.numpy(), initial_gradient, atol=1e-10)
    print('PyTorch自动求导与NumPy梯度一致；设备:', tensor_logits.device)


## 交给你的独立任务

1. 把学习率改成0，解释为什么“有梯度”仍不等于“参数更新”。
2. 自己构造一个只mask单个位置的例子，验证其它位置没有loss贡献。
3. 不用代码回答：这个例子的loss下降为什么不能证明模型能泛化，更不能证明酶有活性？
4. 画出DPLM训练流程，在每一步标注本notebook已经演示了什么、尚未实现什么。
5. 关闭示例，独立实现自己的FASTA长度统计函数。

将预测、错误、修复与答案写进第一周日志。不要把跑通全部单元当作已独立掌握。
